In [2]:
import requests
import polars as pl
import os

# ==============================================================================
# MILESTONE 2.1d: TARGETED BOUNDARY SCHEMA INSPECTION
# Goal: Check specific datasets that mention "boundaries" for SCOTEX/SSEN-S data
# ==============================================================================

API_URL = "https://api.neso.energy/api/3/action/datastore_search"

# Top candidates based on CKAN discovery descriptions mentioning "boundaries"
TARGET_RESOURCES = [
    {
        "name": "Thermal Constraint Costs Data 23-24",
        "id": "75c9c564-af38-4421-a461-a612a6921212",
        "reason": "Description states: 'Out turn system costs for thermal constraints across a number of significant constraint boundaries'"
    },
    {
        "name": "Day Ahead Constraint Flows and Limits",
        "id": "38a18ec1-9e40-465d-93fb-301e80fd1352",
        "reason": "Description states: 'snapshot of the limits and flows at relevant boundaries at day ahead stage'"
    },
    {
        "name": "Thermal Constraint Costs Data 22-23",
        "id": "476b8d39-5eda-425c-9756-73ddfd36dc4d",
        "reason": "Cross-check schema consistency with 23-24 dataset"
    }
]

TARGET_BOUNDARIES = ["SCOTEX", "SSEN-S", "B6", "B2", "SCOT", "SSEN"]

def inspect_resource(resource_info: dict) -> None:
    """Fetch a 5-row sample and inspect schema for boundary attribution."""
    print(f"\n{'='*80}")
    print(f"INSPECTING: {resource_info['name']}")
    print(f"Resource ID: {resource_info['id']}")
    print(f"Reason: {resource_info['reason']}")
    print(f"{'='*80}")
    
    params = {
        "resource_id": resource_info["id"],
        "limit": 5,
        "offset": 0
    }
    
    try:
        response = requests.get(API_URL, params=params)
        response.raise_for_status()
        data = response.json()
        
        if not data.get("success"):
            print(f"  ❌ API Error: {data.get('error')}")
            return
            
        records = data["result"]["records"]
        if not records:
            print("  ⚠️ No records found in this resource.")
            return
            
        df = pl.DataFrame(records)
        print(f"  ✅ Successfully fetched {len(df)} rows.")
        print(f"  📋 Columns: {df.columns}")
        
        # Identify potential boundary/location columns
        boundary_keywords = ["boundary", "constraint", "group", "zone", "region", "name", "id"]
        potential_boundary_cols = [
            col for col in df.columns 
            if any(keyword in col.lower() for keyword in boundary_keywords) and "cost" not in col.lower() and "volume" not in col.lower()
        ]
        
        if not potential_boundary_cols:
            print("  ⚠️ No obvious boundary/location columns found.")
            # Fallback: check all string columns
            potential_boundary_cols = [col for col, dtype in df.schema.items() if dtype == pl.String]
            
        print(f"  🔍 Potential boundary columns to inspect: {potential_boundary_cols}")
        
        for col in potential_boundary_cols:
            print(f"\n  --- Inspecting column: '{col}' ---")
            try:
                # Get unique values, drop nulls, limit to first 15 for readability
                unique_vals = df[col].unique().drop_nulls().head(15).to_list()
                print(f"  Sample unique values: {unique_vals}")
                
                # Check for our target boundaries (case-insensitive)
                unique_str = " ".join(str(v).upper() for v in unique_vals)
                has_scotex = any(target in unique_str for target in ["SCOTEX", "B6"])
                has_ssen_s = any(target in unique_str for target in ["SSEN-S", "SSEN S", "B2"])
                
                if has_scotex:
                    print("  🎯 MATCH FOUND: Contains SCOTEX or B6 reference!")
                if has_ssen_s:
                    print("  🎯 MATCH FOUND: Contains SSEN-S or B2 reference!")
                    
            except Exception as e:
                print(f"  ⚠️ Could not inspect column '{col}': {e}")
                
        # Check for volume/cost columns to ensure we have the right metrics
        metric_cols = [col for col in df.columns if any(k in col.lower() for k in ["volume", "cost", "mwh", "mw"])]
        if metric_cols:
            print(f"\n  📊 Available metric columns: {metric_cols}")
            
    except Exception as e:
        print(f"  ❌ Failed to fetch or process: {e}")

if __name__ == "__main__":
    print("Starting Targeted Boundary Schema Inspection...")
    for res in TARGET_RESOURCES:
        inspect_resource(res)
    
    print(f"\n{'='*80}")
    print("INSPECTION COMPLETE")
    print("NEXT STEP: Review the output above. If a dataset contains both")
    print("'SCOTEX'/'SSEN-S' identifiers AND 'volume'/'cost' metrics, we have")
    print("found our data contract and can proceed to Milestone 2.2.")
    print("If not, we must pivot to the BOALF-to-Boundary mapping strategy.")
    print(f"{'='*80}")

Starting Targeted Boundary Schema Inspection...

INSPECTING: Thermal Constraint Costs Data 23-24
Resource ID: 75c9c564-af38-4421-a461-a612a6921212
Reason: Description states: 'Out turn system costs for thermal constraints across a number of significant constraint boundaries'
  ✅ Successfully fetched 5 rows.
  📋 Columns: ['_id', 'Settlement Date', 'Constraint Group', 'Daily Cost (GBP)']
  🔍 Potential boundary columns to inspect: ['_id', 'Constraint Group']

  --- Inspecting column: '_id' ---
  Sample unique values: [1, 2, 3, 4, 5]

  --- Inspecting column: 'Constraint Group' ---
  Sample unique values: ['SEIMP', 'ESTEX', 'SCOTEX', 'SSE-SP', 'SSHARN']
  🎯 MATCH FOUND: Contains SCOTEX or B6 reference!

  📊 Available metric columns: ['Daily Cost (GBP)']

INSPECTING: Day Ahead Constraint Flows and Limits
Resource ID: 38a18ec1-9e40-465d-93fb-301e80fd1352
Reason: Description states: 'snapshot of the limits and flows at relevant boundaries at day ahead stage'
  ✅ Successfully fetched 5 rows.
 